# EDA with SQL

**Purpose:** Perform the SQL queries required by the IBM capstone lab.

> Starter template: complete this notebook using the corresponding IBM lab. Replace TODO sections with your actual code and outputs.

## Tasks

- [ ] Load the final dataset into the lab SQL environment
- [ ] Run launch-site queries
- [ ] Run payload queries
- [ ] Run success-rate/ranking queries
- [ ] Run time/year analysis
- [ ] Save result tables/screenshots

In [3]:
import pandas as pd
import sqlite3
import os

# ============================================================
# LOAD YOUR EXISTING DATASET
# ============================================================

CSV_FILE = "spacex_dash.csv"

df = pd.read_csv(CSV_FILE)

print("=" * 70)
print("IBM APPLIED DATA SCIENCE CAPSTONE - EDA WITH SQL")
print("=" * 70)

print("\nDataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# Rename to SQL-friendly names
df = df.rename(columns={
    "BoosterVersion": "Booster_Version",
    "PayloadMass": "PAYLOAD_MASS__KG_",
    "LaunchSite": "Launch_Site"
})

# Convert types
df["PAYLOAD_MASS__KG_"] = pd.to_numeric(
    df["PAYLOAD_MASS__KG_"],
    errors="coerce"
)

df["Date"] = df["Date"].astype(str)


# ============================================================
# CREATE DATABASE
# ============================================================

os.makedirs("outputs", exist_ok=True)

connection = sqlite3.connect(
    "outputs/spacex.db"
)

df.to_sql(
    "SPACEXTBL",
    connection,
    if_exists="replace",
    index=False
)

print("\nSQLite database created successfully.")


# ============================================================
# FUNCTION TO RUN QUERY
# ============================================================

def run_query(number, title, query):

    print("\n")
    print("=" * 70)
    print(f"TASK {number}: {title}")
    print("=" * 70)

    print("\nSQL:")
    print(query)

    result = pd.read_sql_query(
        query,
        connection
    )

    print("\nRESULT:")
    display(result)

    return result


# ============================================================
# 1. UNIQUE LAUNCH SITES
# ============================================================

result1 = run_query(
    1,
    "Unique launch sites",

    """
    SELECT DISTINCT Launch_Site
    FROM SPACEXTBL;
    """
)


# ============================================================
# 2. FIVE RECORDS WITH CCA LAUNCH SITE
# ============================================================

result2 = run_query(
    2,
    "Five records with launch site starting with CCA",

    """
    SELECT *
    FROM SPACEXTBL
    WHERE Launch_Site LIKE 'CCA%'
    LIMIT 5;
    """
)


# ============================================================
# 3. TOTAL PAYLOAD
# ============================================================
# Customer column is NOT available in your dataset.
# Therefore we calculate TOTAL payload instead.

result3 = run_query(
    3,
    "Total payload mass",

    """
    SELECT
        SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_KG
    FROM SPACEXTBL;
    """
)


# ============================================================
# 4. AVERAGE PAYLOAD FOR F9 v1.1
# ============================================================

result4 = run_query(
    4,
    "Average payload mass for F9 v1.1",

    """
    SELECT
        AVG(PAYLOAD_MASS__KG_) AS Average_Payload_Mass_KG
    FROM SPACEXTBL
    WHERE Booster_Version LIKE 'F9 v1.1%';
    """
)


# ============================================================
# 5. FIRST SUCCESSFUL LANDING
# ============================================================
# Your dataset uses Outcome instead of Landing_Outcome.

result5 = run_query(
    5,
    "First successful landing",

    """
    SELECT MIN(Date) AS First_Successful_Landing_Date
    FROM SPACEXTBL
    WHERE Outcome LIKE 'True%';
    """
)


# ============================================================
# 6. DRONE-SHIP SUCCESS + PAYLOAD 4000-6000 KG
# ============================================================
# Your Outcome column contains landing information.

result6 = run_query(
    6,
    "Boosters with successful drone-ship landing and payload 4000-6000 kg",

    """
    SELECT DISTINCT Booster_Version
    FROM SPACEXTBL
    WHERE Outcome LIKE 'True (ASDS)%'
    AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;
    """
)


# ============================================================
# 7. SUCCESSFUL AND FAILED LANDING OUTCOMES
# ============================================================

result7 = run_query(
    7,
    "Landing outcome counts",

    """
    SELECT
        Outcome,
        COUNT(*) AS Count
    FROM SPACEXTBL
    GROUP BY Outcome
    ORDER BY Count DESC;
    """
)


# ============================================================
# 8. BOOSTERS WITH MAXIMUM PAYLOAD
# ============================================================

result8 = run_query(
    8,
    "Boosters carrying maximum payload",

    """
    SELECT DISTINCT Booster_Version
    FROM SPACEXTBL
    WHERE PAYLOAD_MASS__KG_ = (
        SELECT MAX(PAYLOAD_MASS__KG_)
        FROM SPACEXTBL
    );
    """
)


# ============================================================
# 9. 2015 FAILED DRONE-SHIP LANDINGS
# ============================================================

result9 = run_query(
    9,
    "2015 failed drone-ship landings",

    """
    SELECT
        substr(Date, 6, 2) AS Month,
        Outcome,
        Booster_Version,
        Launch_Site
    FROM SPACEXTBL
    WHERE Outcome LIKE 'False (ASDS)%'
    AND substr(Date, 1, 4) = '2015';
    """
)


# ============================================================
# 10. RANK LANDING OUTCOMES
# ============================================================

result10 = run_query(
    10,
    "Landing outcome ranking from 2010-06-04 to 2017-03-20",

    """
    SELECT
        Outcome,
        COUNT(*) AS cnt
    FROM SPACEXTBL
    WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY Outcome
    ORDER BY cnt DESC;
    """
)


# ============================================================
# FINISHED
# ============================================================

print("\n")
print("=" * 70)
print("ALL 10 SQL ANALYSIS TASKS COMPLETED")
print("=" * 70)

print("\nDatabase saved at:")
print("outputs/spacex.db")

print("\nUse the displayed SQL queries + results for your presentation.")

IBM APPLIED DATA SCIENCE CAPSTONE - EDA WITH SQL

Dataset loaded successfully!
Rows: 90
Columns: 19

Columns:
['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude', 'Class', 'Year']

SQLite database created successfully.


TASK 1: Unique launch sites

SQL:

    SELECT DISTINCT Launch_Site
    FROM SPACEXTBL;
    

RESULT:


,Launch_Site
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A




TASK 2: Five records with launch site starting with CCA

SQL:

    SELECT *
    FROM SPACEXTBL
    WHERE Launch_Site LIKE 'CCA%'
    LIMIT 5;
    

RESULT:


,FlightNumber,Date,Booster_Version,PAYLOAD_MASS__KG_,Orbit,Launch_Site,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class,Year
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0,2010
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0,2012
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0,2013
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0,2013
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0,2014




TASK 3: Total payload mass

SQL:

    SELECT
        SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_KG
    FROM SPACEXTBL;
    

RESULT:


...



TASK 9: 2015 failed drone-ship landings

SQL:

    SELECT
        substr(Date, 6, 2) AS Month,
        Outcome,
        Booster_Version,
        Launch_Site
    FROM SPACEXTBL
    WHERE Outcome LIKE 'False (ASDS)%'
    AND substr(Date, 1, 4) = '2015';
    

RESULT:


,Month,Outcome,Booster_Version,Launch_Site




TASK 10: Landing outcome ranking from 2010-06-04 to 2017-03-20

SQL:

    SELECT
        Outcome,
        COUNT(*) AS cnt
    FROM SPACEXTBL
    WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY Outcome
    ORDER BY cnt DESC;
    

RESULT:


,Outcome,cnt
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2




ALL 10 SQL ANALYSIS TASKS COMPLETED

Database saved at:
outputs/spacex.db

Use the displayed SQL queries + results for your presentation.


In [ ]:
# TODO: Add the code from your completed IBM lab here.


## Results / Findings

TODO: Add the actual results produced by your notebook. Do not invent numerical values.